# Лабораторная работа 7.

## Разработка рекомендательной системы на Python

[Теория](https://colab.research.google.com/drive/1cmpzRoz0nyPVqKkshK6dO1XhqsOCFo8C?usp=sharing)

**Инструкция:**

- Шаг 1. Постройте рекомендательную систему на примере данных о покупках. Исходные файлы: praktika_7.csv - пользовательские транзакции
   
Имя файла | Ссылка
-- | --
praktika_7.csv | https://github.com/OlesiaAngel/DataAnalitics/blob/main/%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/recomend/praktika_7.csv?raw=true

- Шаг 2. Реализуйте коллаборативную фильтрацию данных на основе пользователей. Используйте модель kNN. Проверить модель на покупателях с customer_id = 4 и customer_id = 21.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Шаг 1. Загрузка данных
url = 'https://github.com/OlesiaAngel/DataAnalitics/blob/main/%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/recomend/praktika_7.csv?raw=true'
df = pd.read_csv(url)
print('Данные о покупках:')
print(df.head(10))
print(f'\nРазмер датасета: {df.shape}')
print(f'\nСтолбцы: {df.columns.tolist()}')
print(f'\nОписание:')
print(df.describe())
print(f'\nПропущенные значения:\n{df.isnull().sum()}')


In [ ]:
# Шаг 2. Создание матрицы пользователь-товар
# Автоматическое определение столбцов
cols = df.columns.tolist()

def find_col(df, keywords):
    for kw in keywords:
        for c in df.columns:
            if kw.lower() in c.lower():
                return c
    return df.columns[0]

customer_col = find_col(df, ['customer', 'user', 'покупат', 'клиент', 'id'])
product_col = find_col(df, ['product', 'item', 'товар', 'продукт', 'sku'])
remaining = [c for c in cols if c not in [customer_col, product_col]]

print(f'Столбец покупателей: {customer_col}')
print(f'Столбец товаров: {product_col}')
print(f'Количество уникальных покупателей: {df[customer_col].nunique()}')
print(f'Количество уникальных товаров: {df[product_col].nunique()}')

if remaining:
    value_col = remaining[0]
    user_item_matrix = df.pivot_table(
        index=customer_col, columns=product_col,
        values=value_col, aggfunc='sum', fill_value=0
    )
else:
    df['purchase'] = 1
    user_item_matrix = df.pivot_table(
        index=customer_col, columns=product_col,
        values='purchase', aggfunc='sum', fill_value=0
    )

print(f'\nМатрица пользователь-товар (форма): {user_item_matrix.shape}')
print(user_item_matrix.head())


In [ ]:
# Шаг 3. Обучение модели kNN (коллаборативная фильтрация на основе пользователей)
sparse_matrix = csr_matrix(user_item_matrix.values)

# k=6: 1 целевой пользователь + 5 похожих
model_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=min(6, user_item_matrix.shape[0]),
    n_jobs=-1
)
model_knn.fit(sparse_matrix)
print('Модель kNN успешно обучена.')
print(f'Параметры модели: {model_knn.get_params()}')


In [ ]:
# Шаг 4. Функция рекомендаций
def get_recommendations(customer_id, user_item_matrix, model_knn, n_recommendations=10):
    """
    Коллаборативная фильтрация на основе пользователей (User-Based CF).
    Возвращает список рекомендованных товаров для заданного покупателя.
    """
    if customer_id not in user_item_matrix.index:
        print(f'Покупатель {customer_id} не найден в данных.')
        return []

    customer_idx = user_item_matrix.index.get_loc(customer_id)
    customer_vector = csr_matrix(
        user_item_matrix.iloc[customer_idx].values.reshape(1, -1)
    )
    distances, indices = model_knn.kneighbors(customer_vector)

    # Похожие пользователи (исключаем самого пользователя)
    similar_users = [
        (user_item_matrix.index[i], distances.flatten()[j])
        for j, i in enumerate(indices.flatten())
        if user_item_matrix.index[i] != customer_id
    ]

    # Товары, которые целевой пользователь уже покупал
    purchased = set(
        user_item_matrix.columns[user_item_matrix.iloc[customer_idx] > 0]
    )

    # Формирование рекомендаций с весовым коэффициентом (сходство)
    recommendations = {}
    for sim_user, dist in similar_users:
        similarity = 1 - dist  # косинусное сходство
        for product in user_item_matrix.columns:
            if user_item_matrix.loc[sim_user, product] > 0 and product not in purchased:
                recommendations[product] = recommendations.get(product, 0) + \
                    similarity * user_item_matrix.loc[sim_user, product]

    sorted_recs = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
    return [item[0] for item in sorted_recs[:n_recommendations]]


def analyze_similar_users(customer_id, user_item_matrix, model_knn):
    """Вывод похожих пользователей для заданного покупателя."""
    if customer_id not in user_item_matrix.index:
        print(f'Покупатель {customer_id} не найден.')
        return
    customer_idx = user_item_matrix.index.get_loc(customer_id)
    customer_vector = csr_matrix(
        user_item_matrix.iloc[customer_idx].values.reshape(1, -1)
    )
    distances, indices = model_knn.kneighbors(customer_vector)
    print(f'Похожие пользователи для customer_id = {customer_id}:')
    for j, (idx, dist) in enumerate(zip(indices.flatten(), distances.flatten())):
        sim_id = user_item_matrix.index[idx]
        if sim_id != customer_id:
            print(f'  customer_id={sim_id:4d} | расстояние={dist:.4f} | сходство={1-dist:.4f}')
print('Функции рекомендаций определены.')


In [ ]:
# Шаг 5. Получение рекомендаций для customer_id = 4 и customer_id = 21
print('=' * 60)
print('РЕКОМЕНДАЦИИ ДЛЯ customer_id = 4')
print('=' * 60)
analyze_similar_users(4, user_item_matrix, model_knn)
recs_4 = get_recommendations(4, user_item_matrix, model_knn)
print(f'\nТоп-10 рекомендованных товаров: {recs_4}')

print('\n' + '=' * 60)
print('РЕКОМЕНДАЦИИ ДЛЯ customer_id = 21')
print('=' * 60)
analyze_similar_users(21, user_item_matrix, model_knn)
recs_21 = get_recommendations(21, user_item_matrix, model_knn)
print(f'\nТоп-10 рекомендованных товаров: {recs_21}')


In [ ]:
# Шаг 6. Визуализация результатов
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (cid, recs) in zip(axes, [(4, recs_4), (21, recs_21)]):
    if cid in user_item_matrix.index:
        purchased_data = user_item_matrix.loc[cid]
        bought = purchased_data[purchased_data > 0]
        n_bought = len(bought)
        ax.set_title(
            f'customer_id = {cid}\n'
            f'Куплено товаров: {n_bought} | '
            f'Рекомендовано: {len(recs)}',
            fontsize=12
        )
        if recs:
            y_pos = range(len(recs))
            ax.barh(y_pos, [i + 1 for i in range(len(recs))],
                    color='steelblue', alpha=0.8)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([f'Товар {r}' for r in recs], fontsize=9)
            ax.set_xlabel('Рейтинг рекомендации (место)')
            ax.invert_yaxis()
            ax.invert_xaxis()
        else:
            ax.text(0.5, 0.5, 'Нет рекомендаций', ha='center',
                    va='center', transform=ax.transAxes, fontsize=14)
    else:
        ax.text(0.5, 0.5, f'customer_id={cid}\nне найден',
                ha='center', va='center', transform=ax.transAxes, fontsize=14)

plt.suptitle('Рекомендательная система (User-Based CF + kNN)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Профиль покупок целевых пользователей
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, cid in zip(axes, [4, 21]):
    if cid in user_item_matrix.index:
        purchased_data = user_item_matrix.loc[cid]
        bought = purchased_data[purchased_data > 0].sort_values(ascending=False)
        ax.bar(range(len(bought)), bought.values, color='coral', alpha=0.8)
        ax.set_title(f'Профиль покупок: customer_id = {cid}\n({len(bought)} уникальных товаров)', fontsize=12)
        ax.set_xlabel('Товары')
        ax.set_ylabel('Количество покупок')
        ax.set_xticks(range(len(bought)))
        ax.set_xticklabels([str(p) for p in bought.index], rotation=45, fontsize=8)
    else:
        ax.text(0.5, 0.5, f'customer_id={cid} не найден',
                ha='center', va='center', transform=ax.transAxes)

plt.suptitle('Профили покупок целевых пользователей', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('\nРекомендательная система построена успешно!')
